# WaferGuard — leakage audit and repaired detector

This notebook repairs the supplied anomaly detector and evaluates it on a later, naturally imbalanced time block. It uses no evaluator-only mask. The model is a review aid; see AUDIT.md before use.

## Reproducibility and guardrails

- Sort by UTC time, then split 70% train / 15% validation / 15% test.
- Fit imputation and scaling on train only.
- Keep PS-L sentinel and firmware categories separate from continuous sensors.
- Use weighted BCE rather than pre-split SMOTE.
- Keep reconstruction and input as [batch, features]; keep logits/labels as [batch].
- Choose the F2 threshold on validation, evaluate test once at its natural prevalence.

In [1]:
from pathlib import Path
import json, sys
sys.path.insert(0, str(Path.cwd() / "src"))
from wafer_guard.train import run_pipeline


In [2]:
result = run_pipeline(
    csv_path="data/secom_telemetry_modified.csv",
    artifact_dir="artifacts",
    epochs=80,
)
print("Best epoch:", result["best_epoch"], "; saved artifacts to artifacts/")


epoch=01 loss=1.3534 val_ap=0.0815 best=0.0815@1
epoch=10 loss=0.0832 val_ap=0.0726 best=0.0835@3
Best epoch: 3 ; saved artifacts to artifacts/


## Held-out results

Numbers below are from the executed training run saved in artifacts/metrics.json. The test set has 9 positive rows, so the operating metrics have wide uncertainty.

In [3]:
with open("artifacts/metrics.json", encoding="utf-8") as f:
    metrics = json.load(f)
summary = {
    "selected_threshold_from_validation": metrics["validation"]["selected_threshold"],
    "validation": {k: metrics["validation"][k] for k in ("rows", "defects", "prevalence", "average_precision")},
    "chronological_test": {k: metrics["test"][k] for k in ("rows", "defects", "prevalence", "average_precision", "roc_auc", "precision", "recall", "f1", "f2", "tn", "fp", "fn", "tp")},
}
summary


{'selected_threshold_from_validation': 0.05295081064105034, 'validation': {'rows': 235, 'defects': 17, 'prevalence': 0.07234042553191489, 'average_precision': 0.08351848706677144}, 'chronological_test': {'rows': 236, 'defects': 9, 'prevalence': 0.038135593220338986, 'average_precision': 0.08421516553953001, 'roc_auc': 0.6162506118453255, 'precision': 0.043478260869565216, 'recall': 0.5555555555555556, 'f1': 0.08064516129032258, 'f2': 0.16556291390728475, 'tn': 117, 'fp': 110, 'fn': 4, 'tp': 5}, 'best_epoch': 3, 'seed': 1337}

## Interpret carefully

The test average precision is 0.0842 at 3.81% prevalence. The validation-selected operating point recalls 5 of 9 defects, but precision is 4.35% (110 false positives). This is not suitable for automatic lot rejection. The score is not probability-calibrated. The dataset and time split are small; acquire later, independent process data before making deployment claims.

## Score an unlabeled sensor file

From the repository root, run: python -m wafer_guard.predict incoming.csv --artifacts artifacts --output scores.csv. The input must contain the same 590 sensor_N columns. label and timestamp are optional for prediction. Output includes a model score, reconstruction error and review flag.